# Lab 4A: Conversational History / Agent Memory

**Environment**: Jupyter in VS Code + Azure OpenAI + Cosmos DB  
**Time**: ~60 min

## Step 0: Initialize connections (prebuilt)

In [ ]:
import os
import uuid
import json
from datetime import datetime, timezone
from azure.identity import AzureCliCredential, get_bearer_token_provider
from azure.cosmos import CosmosClient, PartitionKey
from azure.core.exceptions import HttpResponseError
from openai import OpenAI

# Cosmos DB connection
cosmos_endpoint = os.environ["COSMOS_ENDPOINT"]
credential = AzureCliCredential()
cosmos_client = CosmosClient(cosmos_endpoint, credential)

# Chat memory: Conversations/Messages (pre-provisioned, partition key /sessionId)
chat_db_name = "Conversations"
chat_container_name = "Messages"
chat_database = cosmos_client.get_database_client(chat_db_name)
chat_container = chat_database.get_container_client(chat_container_name)
print(f"Chat memory: {cosmos_endpoint}{chat_db_name}/{chat_container_name}")

# RAG corpus: WorkshopData/Docs (pre-provisioned with vector index on /embedding)
rag_db_name = "WorkshopData"
rag_container_name = "Docs"
rag_database = cosmos_client.get_database_client(rag_db_name)
rag_container = rag_database.get_container_client(rag_container_name)
print(f"RAG corpus:  {cosmos_endpoint}{rag_db_name}/{rag_container_name}")

# Model endpoints use Entra ID with service-specific token audiences.
foundry_endpoint = os.environ["FOUNDRY_ENDPOINT"].rstrip('/')
embeddings_endpoint = os.environ["EMBEDDINGS_ENDPOINT"].rstrip('/')
chat_model = os.environ["COMPLETIONS_MODEL"]
embeddings_model = os.environ["EMBEDDINGS_MODEL"]

foundry_token_provider = get_bearer_token_provider(
    credential, "https://ai.azure.com/.default"
)
foundry_client = OpenAI(
    base_url=f"{foundry_endpoint}/openai/v1/",
    api_key=foundry_token_provider,
)

embeddings_token_provider = get_bearer_token_provider(
    credential, "https://cognitiveservices.azure.com/.default"
)
embeddings_client = OpenAI(
    base_url=f"{embeddings_endpoint}/openai/v1/",
    api_key=embeddings_token_provider,
)

session_id = str(uuid.uuid4())

print(f"\nSession ID:       {session_id}")
print(f"Chat Model:       {chat_model}")
print(f"Embeddings Model: {embeddings_model}")

## Step 1: Seed the RAG corpus (prebuilt)

Populate `WorkshopData/Docs` with general Cosmos DB facts and their embeddings.
Upserts are idempotent, so re-running this cell (or having already run Lab 2E) is safe.

The seed documents themselves live in `rag_seed_docs.json`.

In [ ]:
def embed_text(text: str) -> list[float]:
    resp = embeddings_client.embeddings.create(input=text, model=embeddings_model)
    return resp.data[0].embedding


with open("rag_seed_docs.json", "r", encoding="utf-8") as f:
    rag_seed_docs = json.load(f)

for doc in rag_seed_docs:
    doc["partitionKey"] = "rag"
    doc["embedding"] = embed_text(doc["text"])
    rag_container.upsert_item(body=doc)
    print(f"  Seeded: {doc['title']}")

print(f"\nSeeded {len(rag_seed_docs)} docs into {rag_db_name}/{rag_container_name}")

## Step 2: Chat store message schema (Prebuilt)

Prints the JSON shape used for every chat turn. The `metadata` object carries fields that will be queried in T-SQL in Fabric in Lab 4B (model, latency, token usage, RAG hits, retrieved doc IDs).

In [ ]:
chat_message_schema = {
    "id": "chat_20260524_001",
    "sessionId": "user_session_001",
    "timestamp": "2026-05-24T10:00:00Z",
    "role": "assistant",
    "content": "Hello! How can I help you with Cosmos DB today?",
    "metadata": {
        "model": "phi-4-mini-reasoning",
        "latencyMs": 842,
        "promptTokens": 312,
        "completionTokens": 128,
        "totalTokens": 440,
        "ragHits": 3,
        "retrievedDocIds": ["cosmos_overview", "cosmos_vector_search", "cosmos_request_units"]
    }
}

print(json.dumps(chat_message_schema, indent=2))

# The pre-provisioned 'Messages' container in Cosmos DB works for chat history.
# Partition key path: /sessionId

## Step 2b: Save a conversation turn (Prebuilt)

The `save_chat_turn` helper persists a chat turn to `Conversations/Messages`. It is used by Steps 3, 4, and 5.

In [ ]:
def generate_timestamp() -> str:
    return datetime.now(timezone.utc).isoformat()


def save_chat_turn(session_id: str, role: str, content: str, metadata: dict | None = None) -> dict:
    message = {
        "id": f"{session_id}_{str(uuid.uuid4())[:8]}",
        "sessionId": session_id,
        "role": role,
        "content": content,
        "timestamp": generate_timestamp(),
        "metadata": metadata or {}
    }
    return chat_container.create_item(body=message)


# Test
user_msg = save_chat_turn(session_id, "user", "What is Cosmos DB?")
print(f"Saved user message: {user_msg['id']}")

assistant_msg = save_chat_turn(
    session_id,
    "assistant",
    "Azure Cosmos DB is a globally distributed database service.",
    metadata={"model": chat_model, "latencyMs": 0, "totalTokens": 0}
)
print(f"Saved assistant message: {assistant_msg['id']}")

## Step 3: Retrieve recent messages and run a vector search

Two reads underpin the chat agent: pulling the last N turns for conversational context, and pulling the top-K RAG hits for grounding. The vector search (`retrieve_relevant`) is prebuilt — your job is the conversational read.

Replace the placeholder `query` in `get_recent_messages` with one scoped to the session partition and ordered by `_ts DESC`:

```python
query = f"""SELECT * FROM c WHERE c.sessionId = @sessionId
            ORDER BY c._ts DESC OFFSET 0 LIMIT {count}"""
```

**Expected output**: The recent-message count and contents from Step 2, plus three vector-search hits with their scores.

In [ ]:
def get_recent_messages(session_id: str, count: int = 10) -> list[dict]:
    query = f"""SELECT * FROM c WHERE c.sessionId = @sessionId
                ORDER BY c._ts DESC OFFSET 0 LIMIT {count}"""
    return list(chat_container.query_items(
        query=query,
        parameters=[{"name": "@sessionId", "value": session_id}],
        partition_key=session_id
    ))


def retrieve_relevant(query_text: str, top_k: int = 3) -> list[dict]:
    query_embedding = embed_text(query_text)
    vector_query = f"""SELECT TOP {top_k} c.id, c.title, c.text,
                       VectorDistance(c.embedding, @emb) AS score
                       FROM c WHERE c.partitionKey = 'rag'
                       ORDER BY VectorDistance(c.embedding, @emb)"""
    return list(rag_container.query_items(
        query=vector_query,
        parameters=[{"name": "@emb", "value": query_embedding}],
        partition_key="rag"
    ))


recent = get_recent_messages(session_id, 10)
print(f"Retrieved {len(recent)} recent messages:")
for msg in recent:
    print(f"  [{msg.get('role')}]: {msg.get('content')}")

print("\n=== Retrieval check ===")
for hit in retrieve_relevant("How does vector search work in Cosmos DB?", top_k=3):
    print(f"  - [{hit.get('id')}] score={hit.get('score'):.4f}  {hit.get('title')}")

## Step 4: Build the RAG Chat Agent

The agent already saves the user turn, pulls recent history, runs vector search, calls the LLM, and saves the assistant turn with analytics metadata. Your job is to write the **system prompt** that grounds the model in retrieved context plus chat history.

Replace the placeholder `system_content` in `chat_agent` with one that includes the base prompt, retrieved context, and chat history:

```python
system_content = (
    f"{base_system_prompt}\n\n"
    f"Use the following retrieved context to ground your answer:\n{context_text}\n\n"
    f"Chat history:\n{history}"
)
```

**Expected output**: A grounded answer to the seed question `"How does vector search work in Cosmos DB?"`. With the placeholder prompt, you will see the model answer without retrieved context or history — that is the comparison point.

In [ ]:
import time

base_system_prompt = "You are a helpful assistant specializing in Azure Cosmos DB."


def chat_agent(sid: str, user_message: str) -> str:
    save_chat_turn(sid, "user", user_message)

    context = get_recent_messages(sid, 10)
    ordered = list(reversed(context))
    window = ordered[-6:] if len(ordered) > 6 else ordered
    history = "\n".join(f"{m.get('role')}: {m.get('content')}" for m in window)

    hits = retrieve_relevant(user_message, top_k=3)
    context_text = "\n\n".join(f"{h.get('title')}: {h.get('text')}" for h in hits)

    system_content = (
        f"{base_system_prompt}\n\n"
        f"Use the following retrieved context to ground your answer:\n{context_text}\n\n"
        f"Chat history:\n{history}"
    )

    start = time.perf_counter()
    completion = foundry_client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_message}
        ],
        max_completion_tokens=500
    )
    latency_ms = int((time.perf_counter() - start) * 1000)

    answer = completion.choices[0].message.content
    usage = completion.usage

    metadata = {
        "model": chat_model,
        "latencyMs": latency_ms,
        "promptTokens": getattr(usage, "prompt_tokens", 0),
        "completionTokens": getattr(usage, "completion_tokens", 0),
        "totalTokens": getattr(usage, "total_tokens", 0),
        "ragHits": len(hits),
        "retrievedDocIds": [h.get("id") for h in hits]
    }
    save_chat_turn(sid, "assistant", answer, metadata)
    return answer


user_message = "How does vector search work in Cosmos DB?"
answer = chat_agent(session_id, user_message)
print(f"\nAssistant: {answer}\n")

## Step 5: Chat with your agent

Edit the `user_messages` list below with one or more questions, then run the
cell. Each message is sent to the agent in order and persisted to Cosmos DB
before the next one, so you can:

- Add follow-ups that depend on earlier turns (e.g. *"What is Cosmos DB?"*,
  then *"How does that compare to a relational database?"*) to confirm the
  agent is using prior context.
- Edit the list and re-run the cell as many times as you want — the
  `session_id` stays the same, so history (and the metadata needed for
  Lab 4B) keeps accumulating in the `Messages` container.

In [ ]:
# Edit this list to send one or more messages to the agent.
# Each message runs in order; blank entries are skipped.
# Re-run the cell after editing — history persists across runs.
user_messages = [
    "What is Cosmos DB?",
    "How does that compare to a relational database?",
]

for user_message in user_messages:
    user_message = user_message.strip()
    if not user_message:
        continue
    print(f"You: {user_message}")
    answer = chat_agent(session_id, user_message)
    print(f"Assistant: {answer}")
    print()

## Step 5b: Display conversation history

Pull every turn for this `session_id` back out of Cosmos and print it in order.
This is also a quick check that the agent's writes in Step 5 actually landed —
the same query is what Lab 4B will run against the Fabric mirror.

In [ ]:
final_history = get_recent_messages(session_id, 20)
final_history.reverse()  # get_recent_messages returns newest-first; reverse to show chronological order
print(f"Total messages in session '{session_id}': {len(final_history)}\n")

for msg in final_history:
    role_marker = "User" if msg["role"] == "user" else "Assistant"
    print(f"{role_marker}: {msg['content']}")

print("\n=== COMPLETE ===")